# Milestone 9 — Controlled fine-tuning on Google Colab

This notebook runs the **controlled fine-tuning** workflow (`src/finetune.py`)
on top of the verified baseline, then compares baseline vs fine-tuned on the
**identical validation pipeline** (`src/compare_validation.py`) and stores
every result on **Google Drive** so nothing is lost when the runtime ends.

What it does:

1. Verifies the exact training stack (**TensorFlow 2.20.0 / Keras 3.13.2** —
   the versions of the baseline run and of `.venv-infer`)
2. Clones this repository at a chosen revision (pinnable to a commit SHA)
3. Restores the baseline bundle `model_baseline_20261003_172906.zip` from
   Drive, extracts it to `/content/baseline_model/`, and checks its SHA-256
   against the committed `run_metadata.json`
4. Downloads + inspects RealWaste with the existing `src/` scripts
5. **Validates** the committed split manifests (never regenerates them)
6. Fine-tunes ONLY MobileNetV2 **block_13+** (BatchNorm frozen, backbone
   `training=False`, fresh Adam at 0.00001, max 10 epochs, early stopping)
7. Evaluates BOTH models on validation and applies the documented selection
   rule → `validation_comparison.json`

What it does **not** do (later milestones): test-set evaluation, the
Streamlit app. This milestone reports **validation** metrics only.

> **Honesty note:** this notebook has been validated *locally* (valid JSON,
> every code cell passes a Python syntax check, `tests/test_finetune_notebook.py`
> passes). It has **not been executed in Google Colab**, so it contains **no
> accuracy numbers**. Real numbers must come from an actual run and live in
> `run_metadata.json` / `validation_comparison.json`.

**Runtime:** `Runtime → Change runtime type → T4 GPU` is recommended
(fine-tuning still works on CPU, only slower).


## Step 1 — Inspect the Colab runtime before installing anything

Decide what to install **before** importing TensorFlow:

- The baseline was trained on **TensorFlow 2.20.0 / Keras 3.13.2** (recorded
  in the baseline `run_metadata.json`); `.venv-infer` pins the same versions,
  so this notebook verifies them **exactly** — a different Keras major version
  cannot even load the baseline artifact.
- **Never install `requirements.lock.txt` in Colab.** It contains
  Windows-only pins (e.g. `win32-*` packages) that do not work on Linux.

This cell only reads package metadata; it does **not** import TensorFlow.


In [ ]:
# Step 1: inspect environment (version metadata only — no TensorFlow import).
import platform
from importlib import metadata

TARGET_TF = "2.20.0"
TARGET_KERAS = "3.13.2"


def pkg_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


python_version = platform.python_version()
tf_version = pkg_version("tensorflow")
keras_version = pkg_version("keras")
numpy_version = pkg_version("numpy")

print("Python    :", python_version)
print("TensorFlow:", tf_version or "NOT installed")
print("Keras pkg :", keras_version or "not found as a package")
print("NumPy     :", numpy_version or "NOT installed")
print()
print(f"Required exact versions: TensorFlow {TARGET_TF} / Keras {TARGET_KERAS}.")


## Step 2 — Install only if the versions differ (restart afterwards if you install)

If Step 1 did not report **TensorFlow 2.20.0** and **Keras 3.13.2**, run the
two cells below:

1. the install cell installs the exact pins,
2. the restart cell **restarts the runtime** — this is mandatory after any
   install, otherwise Python keeps using the old already-imported modules and
   you get a confusing half-old/half-new environment.

If the versions already match, you can skip both.


In [ ]:
# Step 2a: decide whether anything needs installing.
from pathlib import Path

INSTALL_MARKER = Path("/content/.ft_install_requested")

needs_install = (tf_version != TARGET_TF) or (keras_version != TARGET_KERAS)

if not needs_install:
    INSTALL_MARKER.unlink(missing_ok=True)
    print(
        f"TensorFlow {tf_version} + Keras {keras_version} match the exact pins "
        "- nothing to install."
    )
    target = None
else:
    target = f"tensorflow=={TARGET_TF} keras=={TARGET_KERAS}"
    INSTALL_MARKER.write_text(target, encoding="utf-8")
    print(f"Will install: {target}")
    print("Run the next cell, then the restart cell.")


In [ ]:
# Step 2b: perform the planned install (skips when the marker is absent).
import subprocess
import sys

if INSTALL_MARKER.exists():
    target = INSTALL_MARKER.read_text(encoding="utf-8")
    print(f"Installing {target} ... (takes a few minutes)")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *target.split()],
        check=True,
    )
    print("Install finished. NOW run the restart cell below.")
else:
    print("No install planned - skipping.")


In [ ]:
# Step 2c: restart the runtime ONLY if something was installed.
# After the restart, notebook variables are cleared: re-run Step 1 and Step 2a
# (they will now report "nothing to install"), then continue from Step 3.
from pathlib import Path

marker = Path("/content/.ft_install_requested")
if marker.exists():
    marker.unlink()
    print("Packages were installed - restarting the runtime now.")
    print("After the restart: re-run Step 1 + Step 2a, then continue at Step 3.")
    get_ipython().kernel.do_shutdown(restart=True)
else:
    print("No restart needed - continue with Step 3.")


## Step 3 — Verify the exact versions actually import

The assertions below are the real gate: fine-tuning and the baseline
comparison only make sense on the stack the baseline was trained with. The
versions printed here are also recorded later in `run_metadata.json`
(`environment` section).


In [ ]:
# Step 3: import and verify the exact versions.
import keras
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)
print("NumPy     :", np.__version__)

assert tf.__version__ == TARGET_TF, (
    f"Expected TensorFlow {TARGET_TF} (the baseline training version), "
    f"got {tf.__version__}. Re-run Step 2."
)
assert keras.__version__ == TARGET_KERAS, (
    f"Expected Keras {TARGET_KERAS} (the baseline training version), "
    f"got {keras.__version__}. Re-run Step 2."
)

gpus = tf.config.list_physical_devices("GPU")
print("GPU(s)    :", gpus if gpus else "none detected (CPU only - fine-tuning slower)")


## Step 4 — Mount Google Drive

Colab's `/content` is **wiped when the runtime ends**. Fine-tuned weights and
comparison reports must live on Drive (or be downloaded before the runtime
dies) — losing a run to a disconnect is the classic Colab mistake.


In [ ]:
# Step 4: mount Drive (safe to re-run).
from google.colab import drive

drive.mount("/content/drive")
print("Drive mounted.")


## Step 5 — Repository, revision, and output locations

- **`REVISION`**: for an exactly reproducible run, pin a full commit SHA
  (run `git rev-parse HEAD` in your local clone). The default `"main"` tracks
  the latest push — convenient, but a later push would change what this
  notebook runs, so record the resolved SHA printed in Step 6.
- Outputs go to `Drive/...-outputs/runs/`, deliberately **outside** the
  clone, so the checkout stays clean (`run_metadata.json` records
  `uncommitted_changes` and a dirty checkout would weaken reproducibility).
- The baseline zip produced by the training notebook must sit in the same
  Drive folder; Step 7 finds it there.


In [ ]:
# Step 5: configuration.
from pathlib import Path

REPO_URL = "https://github.com/snadeem03/ai-waste-classification-assistant.git"
REPO_NAME = "ai-waste-classification-assistant"
# Pin to a full commit SHA for a reproducible run, e.g. REVISION = "<40-hex>".
REVISION = "main"  # tracks the latest push - NOT pinned.

REPO_DIR = Path("/content") / REPO_NAME
BASE_DIR = Path("/content/drive/MyDrive") / f"{REPO_NAME}-outputs"
RUNS_DIR = BASE_DIR / "runs"  # finetune_<stamp>/best_model.keras + metadata + plots
# Download/inspection reports go HERE, not into the clone, so regenerated
# timestamps never overwrite tracked data/metadata/* files.
RUNTIME_METADATA_DIR = BASE_DIR / "runtime_metadata"
# Baseline parent model, extracted from the Drive zip in Step 7.
BASELINE_DIR = Path("/content/baseline_model")
# The comparison report is written straight to Drive (outside the clone),
# so it survives the runtime even if the zipping step never runs.
COMPARISON_OUTPUT = BASE_DIR / "validation_comparison.json"

BASE_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_METADATA_DIR.mkdir(parents=True, exist_ok=True)

print("Repository dir :", REPO_DIR)
print("Checked out    :", REVISION)
print("Persistent dir :", BASE_DIR)
print("Runs dir       :", RUNS_DIR)
print("Runtime reports:", RUNTIME_METADATA_DIR)
print("Baseline dir   :", BASELINE_DIR)
print("Comparison out :", COMPARISON_OUTPUT)


## Step 6 — Clone and check out the revision

Idempotent: re-running fetches updates instead of cloning twice.


In [ ]:
# Step 6: clone (or update) the repository and check out REVISION.
from pathlib import Path
import subprocess

# Normalize BEFORE any path operation. REPO_DIR must be a pathlib.Path
# here even if an earlier cell (or an edit) left it as a plain string -
# otherwise `REPO_DIR / ".git"` raises TypeError: unsupported operand
# type(s) for /: 'str' and 'str'.
REPO_DIR = Path(REPO_DIR)


def git(*args, cwd=None):
    return subprocess.run(
        ["git", *args],
        cwd=str(cwd or REPO_DIR),
        capture_output=True,
        text=True,
        check=True,
    )


if not (REPO_DIR / ".git").exists():
    print("Cloning ...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Repository already present - fetching latest refs.")
    git("fetch", "origin")

git("checkout", REVISION)
if REVISION == "main":
    git("pull", "--ff-only", "origin", "main")
print("Checkout done.")


In [ ]:
# Step 6b: report exactly which commit this run uses.
import subprocess

resolved = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
dirty = subprocess.run(
    ["git", "-C", str(REPO_DIR), "status", "--porcelain"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

print("HEAD            :", resolved)
print("Uncommitted work:", "yes - run is NOT exactly reproducible" if dirty else "no")

if REVISION != "main" and not resolved.startswith(REVISION):
    raise RuntimeError(f"Requested revision {REVISION} but checked out {resolved}.")
if REVISION == "main":
    print('Note: REVISION="main" is unpinned - record the SHA above with your results.')


## Step 7 — Restore the baseline parent model from Drive and verify its checksum

The training notebook saved `model_baseline_20261003_172906.zip` (containing
`best_model.keras`, `class_order.json`, `run_metadata.json`) into the Drive
outputs folder. Fine-tuning loads that file as the **parent**, so two
independent checks run here:

1. the extracted `best_model.keras` must hash to the value recorded in the
   `run_metadata.json` shipped inside the same zip, and
2. the same checksum must appear in the **committed**
   `models/metadata/runs/baseline_20261003_172906/run_metadata.json` of the
   clone — proving the artifact is the verified baseline, not a swapped file.

A mismatch aborts the run; nothing is fine-tuned from an unverified parent.


In [ ]:
# Step 7: extract the baseline zip from Drive and verify its checksum.
import hashlib
import json
import zipfile
from pathlib import Path

BASELINE_RUN_ID = "baseline_20261003_172906"
BASELINE_ZIP = f"model_{BASELINE_RUN_ID}.zip"

candidates = sorted(BASE_DIR.glob(BASELINE_ZIP))
if not candidates:
    raise FileNotFoundError(
        f"No {BASELINE_ZIP} under {BASE_DIR}. Run the training notebook and "
        "download/upload its model zip first."
    )
baseline_zip = candidates[-1]
print("Baseline zip:", baseline_zip)

BASELINE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(baseline_zip) as archive:
    archive.extractall(BASELINE_DIR)

model_path = BASELINE_DIR / "best_model.keras"
if not model_path.exists():
    nested = sorted(BASELINE_DIR.glob("**/best_model.keras"))
    if len(nested) != 1:
        raise FileNotFoundError(
            f"expected exactly one best_model.keras under {BASELINE_DIR}, "
            f"found {len(nested)}"
        )
    model_path = nested[0]


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


# Check 1: the zip's own run_metadata.json vs the file it carries.
bundled = json.loads(
    (model_path.parent / "run_metadata.json").read_text(encoding="utf-8")
)
actual = sha256_file(model_path)
if bundled["model"]["sha256"] != actual:
    raise RuntimeError(
        f"baseline zip is internally inconsistent: file hashes to {actual} "
        f"but its run_metadata.json records {bundled['model']['sha256']}"
    )

# Check 2: the committed report in the clone vs the extracted file.
committed_path = (
    REPO_DIR / "models" / "metadata" / "runs" / BASELINE_RUN_ID / "run_metadata.json"
)
committed = json.loads(committed_path.read_text(encoding="utf-8"))
if committed["model"]["sha256"] != actual:
    raise RuntimeError(
        f"baseline differs from the committed run_metadata.json: file hashes "
        f"to {actual} but {committed_path} records {committed['model']['sha256']}"
    )

BASELINE_MODEL = model_path
print("Baseline model :", BASELINE_MODEL)
print("sha256         :", actual)
print("Checksum verified against the committed run_metadata.json.")


## Step 8 — Download and inspect the dataset

Uses the existing scripts (`src/download_data.py`, `src/inspect_data.py`)
exactly as on a local machine — no notebook-specific download logic. The
download is idempotent: if `data/raw/` already holds the verified archive and
extraction, nothing is re-downloaded (~688 MB). Reports are written to a
runtime directory on Drive via `--metadata-dir`, **not** into the clone —
otherwise every Colab run would rewrite tracked `data/metadata/*` files and
make the checkout dirty.


In [ ]:
# Step 8: download RealWaste (idempotent) and re-run the inspection report.
import json
import subprocess
import sys


def run_script(name, *args):
    cmd = [sys.executable, str(REPO_DIR / "src" / name), *args]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


run_script("download_data.py", "--metadata-dir", str(RUNTIME_METADATA_DIR))
run_script("inspect_data.py", "--metadata-dir", str(RUNTIME_METADATA_DIR))

summary = json.loads(
    (RUNTIME_METADATA_DIR / "inspection_summary.json").read_text(encoding="utf-8")
)
print()
print("selected total (actual) :", summary["actual_selected_total"])
print("selected total (expected):", summary["expected_selected_total"])
print("invalid images           :", summary["image_validation"]["invalid_image_count"])
print("class order              :", summary["class_order"])


## Step 9 — Validate the committed split manifests (never regenerate them)

`data/metadata/*.csv` are **reviewed artifacts**: their SHA-256 checksums and
row counts are recorded in `split_summary.json`. Regenerating them here would
silently redefine the split and break comparability with every earlier
milestone, so instead we verify:

1. checksums of `train.csv` / `validation.csv` / `test.csv` still match the
   recorded ones (byte-integrity of the committed files only — no images and
   no labels are used for any decision here),
2. `train` and `validation` load through `src/data_pipeline.py` with the
   expected row counts — this is exactly the loader fine-tuning will use.

Fine-tuning later opens **only** `train.csv` + `validation.csv`;
`run_metadata.json` records `data.test_manifest_used: false` to prove it.


In [ ]:
# Step 9: verify committed manifests against split_summary.json.
import hashlib
import json
import sys

sys.path.insert(0, str(REPO_DIR / "src"))
import data_pipeline


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


metadata_dir = REPO_DIR / "data" / "metadata"
summary = json.loads((metadata_dir / "split_summary.json").read_text(encoding="utf-8"))

for name in ("train.csv", "validation.csv", "test.csv"):
    actual = sha256_file(metadata_dir / name)
    expected = summary["manifest_checksums_sha256"][name]
    if actual != expected:
        raise RuntimeError(f"{name} checksum mismatch: {actual} != {expected}")
    print(f"{name}: checksum OK ({actual[:12]}...)")

splits = data_pipeline.load_splits(
    metadata_dir=metadata_dir,
    path_root=REPO_DIR,
    splits=("train", "validation"),
)
for split in ("train", "validation"):
    count = len(splits[split])
    expected = summary["split_totals"][split]
    if count != expected:
        raise RuntimeError(f"{split}: {count} rows != expected {expected}")
    print(f"{split}: {count} rows OK")

print("All committed manifests validated - none were regenerated.")


## Step 10 — Accelerator check

Record what hardware this run actually gets (also captured automatically in
`run_metadata.json` → `environment`).


In [ ]:
# Step 10: accelerator details.
import subprocess

try:
    result = subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=True, timeout=60
    )
    print(result.stdout)
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    print("nvidia-smi unavailable:", exc)

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow GPUs:", gpus if gpus else "none (CPU only)")


## Step 11 — The fine-tuning policy and the selection rule (fixed BEFORE running)

**Fine-tuning policy** (`src/finetune.py`, verified again by its tests):

- Only MobileNetV2 layers from **`block_13` onward** are unfrozen (index-based
  selection — the first block is named `expanded_conv`, not `block_1`);
  earlier blocks stay frozen and **every BatchNorm layer stays frozen**.
- The backbone keeps being called with **`training=False`** — this is read
  back from the loaded graph and fine-tuning **refuses to start** if the flag
  is not `False`, so BatchNorm running statistics never change.
- A **fresh Adam at learning rate 0.00001** (the baseline's optimizer state is
  deliberately not reused), sparse cross-entropy, at most **10 epochs**,
  `EarlyStopping` + best-checkpoint on `val_loss` (patience 3).
- Same `train.csv` + `validation.csv` as the baseline, same seed 42 and
  batch size 16 from `configs/training.json`. The test manifest is never
  opened. The parent file is checksummed before **and** after training.
- A separate `finetune_<timestamp>` run directory is created — it can never
  overwrite the baseline.

**Selection rule** (documented here and in `src/compare_validation.py`
*before* any comparison is run; measured metrics can never change it):

1. If the fine-tuned model's **macro F1** is strictly higher than the
   baseline's → select the fine-tuned model.
2. If macro F1 is equal, select the fine-tuned model only when its
   **validation accuracy** is strictly higher.
3. Otherwise (macro F1 lower, or both equal) → keep the baseline.

Both models are evaluated on the **identical** validation pipeline (same
manifest, same deterministic order, same image size and batch size). The rule
uses **validation** metrics only — no test-set access in this milestone.


In [ ]:
# Step 11: controlled fine-tuning (block_13+, fresh Adam 1e-5, validation only).
import json
import os
import subprocess
import sys

config = json.loads(
    (REPO_DIR / "configs" / "training.json").read_text(encoding="utf-8")
)
SEED = config["seed"]

env = dict(os.environ, PYTHONHASHSEED=str(SEED))
cmd = [
    sys.executable,
    str(REPO_DIR / "src" / "finetune.py"),
    "--parent-model",
    str(BASELINE_MODEL),
    "--output-dir",
    str(RUNS_DIR),
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(REPO_DIR), env=env)
if result.returncode != 0:
    raise RuntimeError("Fine-tuning failed - see the log output above.")

run_dirs = [p for p in RUNS_DIR.glob("finetune_*") if p.is_dir()]
if not run_dirs:
    raise RuntimeError(f"No finetune_* run directory found under {RUNS_DIR}")
RUN_DIR = max(run_dirs, key=lambda p: p.stat().st_mtime)
print("Run directory:", RUN_DIR)


## Step 12 — Compare baseline vs fine-tuned on the identical validation pipeline

Runs `src/compare_validation.py` as a subprocess: both models go through the
same deterministic validation pass, the metrics are computed in pure NumPy,
the selection rule above is applied, and the report is written to
`validation_comparison.json` on Drive.

If the fine-tuned model were missing, the report would be written with status
`pending` and **no** metrics — nothing is ever invented. The test split is
not opened by this script.


In [ ]:
# Step 12: validation-only comparison + model selection (no test-set access).
import json
import os
import subprocess
import sys

env = dict(os.environ, PYTHONHASHSEED=str(SEED))
cmd = [
    sys.executable,
    str(REPO_DIR / "src" / "compare_validation.py"),
    "--baseline-model",
    str(BASELINE_MODEL),
    "--finetune-model",
    str(RUN_DIR / "best_model.keras"),
    "--config",
    str(REPO_DIR / "configs" / "training.json"),
    "--output",
    str(COMPARISON_OUTPUT),
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(REPO_DIR), env=env)
if result.returncode != 0:
    raise RuntimeError("Comparison failed - see the log output above.")

report = json.loads(COMPARISON_OUTPUT.read_text(encoding="utf-8"))
print()
print("Status           :", report["status"])
print("Decision         :", report["decision"]["selected"])
print("Reason           :", report["decision"]["reason"])
if report["status"] == "complete":
    print("Baseline macro F1  :",
          report["evaluation"]["baseline"]["metrics"]["macro_f1"])
    print("Fine-tuned macro F1:",
          report["evaluation"]["finetuned"]["metrics"]["macro_f1"])
print("Test manifest used :", report["evaluation"]["test_manifest_used"])


## Step 13 — Show the curves and the recorded results

Everything below is **read from the run's files** (`history.csv`,
`run_metadata.json`) — numbers are never typed by hand. These are
**validation** metrics; the test split was not touched.


In [ ]:
# Step 13: plot curves + print the recorded run metadata.
import csv
import json

import matplotlib.pyplot as plt

metadata = json.loads((RUN_DIR / "run_metadata.json").read_text(encoding="utf-8"))
with open(RUN_DIR / "history.csv", newline="", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

epochs = [int(row["epoch"]) for row in rows]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [float(r["loss"]) for r in rows], label="train")
axes[0].plot(epochs, [float(r["val_loss"]) for r in rows], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(epochs, [float(r["accuracy"]) for r in rows], label="train")
axes[1].plot(epochs, [float(r["val_accuracy"]) for r in rows], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

best = metadata["training"]["best"]
policy = metadata["fine_tuning_policy"]
print("Run id            :", metadata["run_id"])
print("Parent run        :", metadata["parent"].get("run_id"))
print("Parent sha256     :", metadata["parent"]["sha256"][:16] + "...",
      "(unchanged:", str(metadata["parent"]["unchanged_after_training"]) + ")")
print("Epochs completed  :", metadata["configuration"]["epochs_completed"],
      "(max", str(metadata["configuration"]["epochs_requested"]) + ")")
print("Best epoch        :", best["epoch"])
print("Validation loss   :", round(best["val_loss"], 4))
print("Validation accuracy:", round(best["val_accuracy"], 4))
print("Learning rate     :", metadata["configuration"]["learning_rate"])
print("Unfrozen from     :", policy["start_layer"],
      f"({len(policy['backbone_trainable_layers'])} backbone layers trainable,",
      f"{policy['backbone_frozen_layers_count']} frozen,",
      f"{policy['batch_norm_layers']} BatchNorm frozen)")
print("Backbone training :", policy["backbone_call_training"])
print("Class order       :", metadata["class_order"])
print("Git commit        :", metadata["git"]["commit"])
print("Uncommitted files :", metadata["git"]["uncommitted_changes"])
print("Model sha256      :", metadata["model"]["sha256"][:16] + "...")
print("Test set used     :", metadata["data"]["test_manifest_used"])


## Step 14 — Package the small reports + the model for transport

- The small report files (`run_metadata.json`, `class_order.json`,
  `history.csv`, `environment_freeze.txt`, `plots/`) were already exported
  into the clone at `models/metadata/runs/<run_id>/` (never a model binary).
- Then we zip (a) those reports, (b) `best_model.keras`, and (c) the
  comparison report into Drive so all three can be downloaded. The model zip
  stays out of Git (`.gitignore` blocks `*.keras` and `*.zip`); the report
  files may be committed to Git locally after download.


In [ ]:
# Step 14: zip the reports, the model, and the comparison report to Drive.
import zipfile
from pathlib import Path

reports_dir = REPO_DIR / "models" / "metadata" / "runs" / RUN_DIR.name
if not reports_dir.is_dir():
    raise FileNotFoundError(f"expected exported small reports at {reports_dir}")

import shutil

reports_zip = shutil.make_archive(
    str(BASE_DIR / f"reports_{RUN_DIR.name}"), "zip", reports_dir
)

model_zip = str(BASE_DIR / f"model_{RUN_DIR.name}.zip")
with zipfile.ZipFile(model_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(RUN_DIR / "best_model.keras", arcname="best_model.keras")
    archive.write(RUN_DIR / "class_order.json", arcname="class_order.json")
    archive.write(RUN_DIR / "run_metadata.json", arcname="run_metadata.json")

comparison_zip = str(BASE_DIR / "validation_comparison.zip")
with zipfile.ZipFile(comparison_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(COMPARISON_OUTPUT, arcname="validation_comparison.json")

print("Reports zip    :", reports_zip)
print("Model zip      :", model_zip)
print("Comparison zip :", comparison_zip)
print()
print("Download from the Colab file browser (folder icon, left panel):")
print("  /content/drive/MyDrive/" + BASE_DIR.name + "/")

# Optional: trigger a browser download instead of using the file browser.
DOWNLOAD_IN_BROWSER = False
if DOWNLOAD_IN_BROWSER:
    from google.colab import files

    files.download(reports_zip)
    files.download(model_zip)
    files.download(comparison_zip)


## Step 15 — Bring the results home and verify

Download `reports_<run>.zip`, `model_<run>.zip`, and
`validation_comparison.zip` from `Drive/...-outputs/`, then on your machine:

```powershell
# restore the Git-trackable reports of the fine-tune run
Expand-Archive .\reports_finetune_<stamp>.zip `
  -DestinationPath .\models\metadata\runs\finetune_<stamp>

# keep the weights next to the other ignored binaries (stays out of Git)
Expand-Archive .\model_finetune_<stamp>.zip -DestinationPath .\models\finetune_<stamp>

# the model-selection report (its canonical home is models/metadata/comparison/)
Copy-Item .\validation_comparison.json `
  .\models\metadata\comparison\validation_comparison.json
```

**Verification commands — run before ever claiming "the model loads" locally:**

```powershell
# the fine-tuned artifact was saved by Keras 3 -> use the .venv-infer env
.venv-infer\Scripts\python.exe -c "import tensorflow as tf; m = tf.keras.models.load_model(r'models\finetune_<stamp>\best_model.keras'); print(m.output_shape)"
```

### Reproducibility limits (read before comparing numbers)

- Each run records its own environment (`environment_freeze.txt` and
  `run_metadata.json` → `environment`). A Colab runtime is **not** the local
  Windows/Python 3.11 environment; different GPUs and library builds change
  floating-point results slightly even with seed 42.
- Early stopping makes the epoch count depend on data + hardware — expect
  different stopping epochs across environments.
- `REVISION = "main"` is unpinned unless you set a commit SHA.
- The selection rule compares the two models **on validation only**; a
  different validation pass (other image size, other batch order) would be a
  different comparison — always use `src/compare_validation.py`.

### Honesty statements

- This notebook has **not been executed in Google Colab**. It was validated
  locally: valid JSON, every code cell passes a Python syntax check, and
  `tests/test_finetune_notebook.py` passes. It deliberately contains **no
  accuracy numbers**.
- This milestone reports **validation** metrics only; the test split is
  evaluated in a later milestone, and neither script here can open it.
- Any number you later read in `run_metadata.json` or
  `validation_comparison.json` must come from that run's own files — never
  from a different run or another project.
